# STEP 1: Install **Libraries**

In [ ]:
!pip install transformers datasets scikit-learn torch -q

# STEP 2: Import **Libraries**

In [ ]:
import pandas as pd
import numpy as np
import torch
import re

from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer

import matplotlib.pyplot as plt
import seaborn as sns

# STEP 3: Load **Dataset**

In [ ]:
dataset = load_dataset("imdb")

# STEP 4: Convert to **DataFrame**

In [ ]:
train_df = pd.DataFrame(dataset['train'])
test_df = pd.DataFrame(dataset['test'])

# STEP 5: **Preprocessing**

In [ ]:
# Reduce size (faster training)
train_df = train_df.sample(6000, random_state=42)
test_df = test_df.sample(1500, random_state=42)

def clean_text(text):
    text = text.lower()
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'[^a-zA-Z ]', '', text)
    return text

train_df['text'] = train_df['text'].apply(clean_text)
test_df['text'] = test_df['text'].apply(clean_text)

# STEP 6: Train / Validation **Split**

In [ ]:
train_texts, val_texts, train_labels, val_labels = train_test_split(
    train_df['text'], train_df['label'], test_size=0.1, random_state=42
)

# STEP 7: **Tokenization**

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def tokenize_function(texts):
    return tokenizer(list(texts), padding=True, truncation=True, max_length=128)

train_encodings = tokenize_function(train_texts)
val_encodings = tokenize_function(val_texts)
test_encodings = tokenize_function(test_df['text'])

# STEP 8: PyTorch **Dataset**

In [ ]:
class IMDbDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels.tolist()

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = IMDbDataset(train_encodings, train_labels)
val_dataset = IMDbDataset(val_encodings, val_labels)
test_dataset = IMDbDataset(test_encodings, test_df['label'])

# STEP 9: Load **Model**

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

# STEP 10: Training **Arguments**

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    num_train_epochs=2
)

# STEP 11: Metrics **Function**

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall
    }

# STEP 12: **Trainer**

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

# STEP 13: Train **Model**

In [ ]:
trainer.train()

# STEP 14: Evaluate on **Validation**

In [ ]:
trainer.evaluate()

# STEP 15: **Predictions**

In [ ]:
predictions = trainer.predict(test_dataset)

y_pred = np.argmax(predictions.predictions, axis=1)
y_true = test_df['label'].values

# **STEP 16: Evaluation Metrics**

In [ ]:
precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='binary')
accuracy = accuracy_score(y_true, y_pred)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)

print("\nClassification Report:\n")
print(classification_report(y_true, y_pred))

# STEP 17: Confusion **Matrix**

In [ ]:
cm = confusion_matrix(y_true, y_pred)

plt.figure()
sns.heatmap(cm, annot=True, fmt='d')
plt.title("Confusion Matrix - BERT IMDB")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

# EXPERIMENT 1: Freeze **BERT**

In [ ]:
for param in model.base_model.parameters():
    param.requires_grad = False

trainer.train()

# **EXPERIMENT 2: Fine-tune Last 2 Layers**

In [ ]:
for name, param in model.named_parameters():
    if "encoder.layer.10" in name or "encoder.layer.11" in name:
        param.requires_grad = True
    else:
        param.requires_grad = False

trainer.train()

### Final Conclusion

- Full fine-tuning of BERT achieved the highest accuracy among all experiments.
- Freezing BERT layers reduced training time but led to lower model performance.
- Fine-tuning only the last two layers provided a good balance between speed and accuracy.
- Overall, BERT proved to be highly effective for sentiment classification tasks on the IMDB dataset.